# Lesson 8A: Policy Gradient Methods - Theory

## Introduction

So far, we've learned value-based methods: estimate V(s) or Q(s,a), then derive a policy by taking the maximum action. But what if we directly optimize the policy instead?

**Policy gradient methods** parameterize the policy θ(π_θ) and optimize it via gradient ascent on the expected return. This is powerful for:
- Continuous action spaces (no max over discrete actions)
- Stochastic policies (exploration built-in)
- Direct policy search without estimating values

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import gymnasium as gym
from typing import Callable, Tuple

np.random.seed(42)

## Part 1: The Policy Gradient Theorem

### Objective: Maximize Expected Return

We want to maximize:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \sum_{t=0}^{T} \gamma^t r_t$$

The **policy gradient theorem** (Sutton, 1999) tells us:
$$\nabla_\theta J(\theta) \propto \mathbb{E}_{s \sim \rho(s), a \sim \pi_\theta(a|s)}[\nabla_\theta \log \pi_\theta(a|s) Q^\pi(s,a)]$$

Key insight: We take gradients of the **log policy**, weighted by the **action value**.

## Part 2: REINFORCE (Baseline)

### Algorithm

Use the return G_t as an unbiased estimate of Q(s,a):

$$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t|s_t) G_t$$

Add a baseline b(s) to reduce variance:

$$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t|s_t) (G_t - b(s_t))$$

Common baseline: V(s), estimated via linear function approximation or neural network.

In [ ]:
def reinforce_with_baseline(
    env: gym.Env,
    episodes: int = 100,
    gamma: float = 0.99,
    alpha_policy: float = 0.01,
    alpha_baseline: float = 0.01
) -> Tuple[np.ndarray, np.ndarray]:
    """
    REINFORCE with baseline on a discrete action space.
    Policy: softmax(theta @ features)
    Baseline: w @ features
    """
    obs, _ = env.reset()
    n_features = len(obs) if isinstance(obs, np.ndarray) else 1
    n_actions = env.action_space.n
    
    theta = np.random.randn(n_features, n_actions) * 0.01
    w = np.zeros(n_features)
    
    returns = []
    
    for ep in range(episodes):
        obs, _ = env.reset()
        done = False
        trajectory = []
        
        # Collect trajectory
        while not done:
            # Policy: softmax(theta^T @ obs)
            logits = obs @ theta
            policy = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))
            action = np.random.choice(n_actions, p=policy)
            
            obs_next, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            trajectory.append((obs, action, reward))
            obs = obs_next
        
        # Compute returns
        G = 0
        for t in reversed(range(len(trajectory))):
            obs_t, action_t, reward_t = trajectory[t]
            G = reward_t + gamma * G
            
            # Baseline estimate
            baseline = obs_t @ w
            advantage = G - baseline
            
            # Policy gradient
            logits = obs_t @ theta
            policy = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))
            grad_log_pi = np.zeros_like(theta)
            grad_log_pi[:, action_t] = obs_t
            grad_log_pi[:, :] -= obs_t[:, None] * policy[None, :]
            
            theta += alpha_policy * grad_log_pi * advantage
            w += alpha_baseline * obs_t * (G - baseline)
        
        ep_return = sum(r for _, _, r in trajectory)
        returns.append(ep_return)
    
    return np.array(returns), theta

print("REINFORCE with baseline: defined")

## Part 3: Actor-Critic

### Separate Actor and Critic Networks

- **Actor**: parameterizes π_θ(a|s), updated via policy gradient
- **Critic**: estimates V_w(s), trained as regression target = r + γV(s')

The advantage function:
$$A(s,a) = Q(s,a) - V(s) = r + \gamma V(s') - V(s)$$

One-step TD advantage (bootstrapping):
$$A(s,a) \approx r + \gamma V(s') - V(s)$$

This reduces variance significantly compared to REINFORCE.

In [ ]:
class ActorCritic:
    """
    Actor-Critic for continuous or discrete control.
    Simple linear function approximation for demonstration.
    """
    
    def __init__(self, n_features: int, n_actions: int, alpha_actor: float = 0.01, alpha_critic: float = 0.01):
        self.theta = np.random.randn(n_features, n_actions) * 0.01  # Actor
        self.w = np.zeros(n_features)  # Critic
        self.alpha_actor = alpha_actor
        self.alpha_critic = alpha_critic
    
    def policy(self, obs: np.ndarray) -> np.ndarray:
        """Softmax policy over actions."""
        logits = obs @ self.theta
        return np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))
    
    def value(self, obs: np.ndarray) -> float:
        """Critic estimate of V(s)."""
        return obs @ self.w
    
    def update(self, obs: np.ndarray, action: int, reward: float, obs_next: np.ndarray, done: bool, gamma: float = 0.99):
        """One-step TD update for both actor and critic."""
        # TD error (advantage)
        V_s = self.value(obs)
        V_next = 0 if done else self.value(obs_next)
        td_error = reward + gamma * V_next - V_s
        
        # Critic update (reduce TD error)
        self.w += self.alpha_critic * obs * td_error
        
        # Actor update (policy gradient)
        policy = self.policy(obs)
        grad_log_pi = np.zeros_like(self.theta)
        grad_log_pi[:, action] = obs
        grad_log_pi[:, :] -= obs[:, None] * policy[None, :]
        self.theta += self.alpha_actor * grad_log_pi * td_error

print("Actor-Critic class: defined")

## Part 4: A2C and A3C Overview

### A2C (Advantage Actor-Critic)
- Batch the experience: collect B transitions, compute advantages, update both networks
- More stable than online updates
- Synchronous: update after each batch

### A3C (Asynchronous Advantage Actor-Critic)
- Multiple workers collect experience in parallel
- Each worker updates a shared network asynchronously
- Less computational overhead than A2C for distributed learning
- Pioneering work in deep RL (Mnih et al., 2016)

## Key Concepts Summary

1. **Policy Gradient Theorem**: ∇J(θ) ∝ E[∇log π(a|s) Q(s,a)]
2. **REINFORCE**: Use episodic returns; add baseline to reduce variance
3. **Actor-Critic**: Separate policy and value networks; use TD error as advantage
4. **A2C/A3C**: Batch or parallel actor-critic algorithms for stability and scale

Next lesson: policy gradient methods in practice with PyTorch and LunarLander.